# Illustrate — Interactive Notebook

Goodsell-style molecular illustration with live controls.  
Run all cells, then use the widgets below to explore.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from PIL import Image
import numpy as np

from illustrate import RenderParams, Transform, render, write_png
from illustrate.fetch import fetch_pdb
from illustrate.presets import PRESET_NAMES, render_params_from_preset

In [ ]:
# ── State ──
_pdb_path = None
_last_result = None

# ── Widgets ──
pdb_input = widgets.Text(value="2hhb", placeholder="PDB ID (e.g. 2hhb)", description="PDB ID:")
fetch_btn = widgets.Button(description="Fetch", button_style="primary", layout=widgets.Layout(width="80px"))
preset_dd = widgets.Dropdown(options=PRESET_NAMES, value="Default", description="Preset:")

scale_sl   = widgets.FloatSlider(value=12.0, min=1, max=40, step=0.5, description="Scale:")
xrot_sl    = widgets.FloatSlider(value=0,    min=-180, max=180, step=1, description="X rotation:")
yrot_sl    = widgets.FloatSlider(value=0,    min=-180, max=180, step=1, description="Y rotation:")
zrot_sl    = widgets.FloatSlider(value=90,   min=-180, max=180, step=1, description="Z rotation:")
shadows_cb = widgets.Checkbox(value=True, description="Shadows")
outlines_cb = widgets.Checkbox(value=True, description="Outlines")

render_btn = widgets.Button(description="Render", button_style="success")
save_btn   = widgets.Button(description="Save PNG", button_style="")
status_lbl = widgets.Label(value="Enter a PDB ID and click Fetch.")
output_area = widgets.Output()

# ── Layout ──
fetch_row = widgets.HBox([pdb_input, fetch_btn])
toggle_row = widgets.HBox([shadows_cb, outlines_cb])
action_row = widgets.HBox([render_btn, save_btn, status_lbl])

controls = widgets.VBox([
    fetch_row,
    preset_dd,
    scale_sl, xrot_sl, yrot_sl, zrot_sl,
    toggle_row,
    action_row,
])

# ── Callbacks ──
def on_fetch(_):
    global _pdb_path
    pdb_id = pdb_input.value.strip()
    if not pdb_id:
        status_lbl.value = "Enter a PDB ID."
        return
    status_lbl.value = f"Fetching {pdb_id}..."
    try:
        _pdb_path = str(fetch_pdb(pdb_id))
        status_lbl.value = f"Loaded {pdb_id} — click Render."
    except Exception as e:
        status_lbl.value = f"Fetch failed: {e}"

def on_render(_):
    global _last_result
    if _pdb_path is None:
        status_lbl.value = "Fetch a PDB first."
        return
    status_lbl.value = "Rendering..."
    try:
        from illustrate.types import WorldParams, OutlineParams
        params = render_params_from_preset(preset_dd.value, _pdb_path)
        params = RenderParams(
            pdb_path=params.pdb_path,
            rules=params.rules,
            transform=Transform(
                scale=scale_sl.value,
                rotations=[("z", zrot_sl.value), ("y", yrot_sl.value), ("x", xrot_sl.value)],
                autocenter="auto",
            ),
            world=WorldParams(
                background=params.world.background, fog_color=params.world.fog_color,
                fog_front=params.world.fog_front, fog_back=params.world.fog_back,
                shadows=shadows_cb.value, shadow_strength=params.world.shadow_strength,
                shadow_angle=params.world.shadow_angle, shadow_min_z=params.world.shadow_min_z,
                shadow_max_dark=params.world.shadow_max_dark,
                width=params.world.width, height=params.world.height,
            ),
            outlines=OutlineParams(enabled=outlines_cb.value) if not outlines_cb.value else params.outlines,
        )
        _last_result = render(params)
        img = Image.fromarray(_last_result.rgb)
        with output_area:
            clear_output(wait=True)
            display(img)
        status_lbl.value = f"Done — {_last_result.width}x{_last_result.height}"
    except Exception as e:
        status_lbl.value = f"Error: {e}"

def on_save(_):
    if _last_result is None:
        status_lbl.value = "Nothing to save — render first."
        return
    write_png("illustrate_output.png", _last_result.rgb)
    status_lbl.value = "Saved illustrate_output.png"

fetch_btn.on_click(on_fetch)
render_btn.on_click(on_render)
save_btn.on_click(on_save)

display(controls, output_area)